# EcoPackAI — Module 2: Data Cleaning & Feature Engineering
**Milestone 1 | Week 1–2**

This notebook covers:
1. Import data from PostgreSQL
2. Data inspection & validation
3. Data cleaning (nulls, duplicates, outliers)
4. Encoding categorical features
5. Normalization of numerical features
6. Feature Engineering:
   - CO₂ Impact Index
   - Cost Efficiency Index
   - Material Suitability Score
7. Final data quality report

## 1. Install & Import Libraries

In [ ]:
# Install if not already available
# !pip install psycopg2-binary sqlalchemy pandas numpy scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sqlalchemy import create_engine
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✅ All libraries imported successfully.")

## 2. Connect to PostgreSQL Database

In [ ]:
# ── Update these credentials to match your setup ──────────────
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "ecopackai"
DB_USER = "ecopackai_user"
DB_PASS = "EcoPack@2024"
# ────────────────────────────────────────────────────────────

connection_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_url)

# Test connection
with engine.connect() as conn:
    print("✅ Connected to PostgreSQL database:", DB_NAME)

## 3. Load Data from Database

In [ ]:
# Load all three tables
df_materials   = pd.read_sql("SELECT * FROM packaging_materials", engine)
df_categories  = pd.read_sql("SELECT * FROM product_categories", engine)
df_mapping     = pd.read_sql("SELECT * FROM material_category_mapping", engine)

print(f"📦 packaging_materials      : {df_materials.shape}")
print(f"🏷️  product_categories       : {df_categories.shape}")
print(f"🔗 material_category_mapping: {df_mapping.shape}")

## 4. Initial Data Inspection

In [ ]:
# Preview materials table
print("── First 5 rows ──────────────────────────────────────────────")
display(df_materials.head())

print("\n── Data Types & Non-Null Counts ──────────────────────────────")
display(df_materials.info())

print("\n── Basic Statistics ──────────────────────────────────────────")
display(df_materials.describe().T)

## 5. Data Cleaning

### 5.1 Check Missing Values

In [ ]:
missing = df_materials.isnull().sum()
missing_pct = (missing / len(df_materials) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print("✅ No missing values found in packaging_materials.")
else:
    print("⚠️ Missing values detected:")
    display(missing_df)
    # Fill numerical with median, categorical with mode
    num_cols = df_materials.select_dtypes(include='number').columns
    cat_cols = df_materials.select_dtypes(include='object').columns
    df_materials[num_cols] = df_materials[num_cols].fillna(df_materials[num_cols].median())
    df_materials[cat_cols] = df_materials[cat_cols].fillna(df_materials[cat_cols].mode().iloc[0])
    print("✅ Missing values filled (numerical → median, categorical → mode).")

### 5.2 Check & Remove Duplicates

In [ ]:
dups = df_materials.duplicated().sum()
print(f"Duplicate rows found: {dups}")

if dups > 0:
    df_materials.drop_duplicates(inplace=True)
    df_materials.reset_index(drop=True, inplace=True)
    print(f"✅ Duplicates removed. Remaining rows: {len(df_materials)}")
else:
    print("✅ No duplicates found.")

### 5.3 Outlier Detection (IQR Method)

In [ ]:
num_cols = ['strength_rating', 'weight_capacity_kg', 'biodegradability_score',
            'co2_emission_score', 'recyclability_percentage',
            'cost_per_kg_usd', 'decomposition_time_days', 'moisture_resistance']

outlier_report = {}
for col in num_cols:
    Q1  = df_materials[col].quantile(0.25)
    Q3  = df_materials[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = df_materials[(df_materials[col] < lower) | (df_materials[col] > upper)]
    outlier_report[col] = {'Lower Bound': round(lower, 2),
                           'Upper Bound': round(upper, 2),
                           'Outlier Count': len(outliers)}

outlier_df = pd.DataFrame(outlier_report).T
display(outlier_df)

# Cap outliers using IQR (Winsorization) - preserve all rows
for col in num_cols:
    Q1  = df_materials[col].quantile(0.25)
    Q3  = df_materials[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    df_materials[col] = df_materials[col].clip(lower=lower, upper=upper)

print("\n✅ Outliers capped using IQR Winsorization (no rows dropped).")

### 5.4 Feature Distribution Plots

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("Distribution of Numerical Features (Post-Cleaning)", fontsize=14, fontweight='bold')

for ax, col in zip(axes.flatten(), num_cols):
    ax.hist(df_materials[col], bins=25, color='teal', edgecolor='white', alpha=0.85)
    ax.set_title(col.replace('_', ' ').title(), fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('Count')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Distribution plot saved as 'feature_distributions.png'")

## 6. Encoding Categorical Features

In [ ]:
# ── Label Encoding for material_type and source_type ──────────
le_type   = LabelEncoder()
le_source = LabelEncoder()

df_materials['material_type_enc'] = le_type.fit_transform(df_materials['material_type'])
df_materials['source_type_enc']   = le_source.fit_transform(df_materials['source_type'])

# Save mappings for reference
type_mapping   = dict(zip(le_type.classes_,   le_type.transform(le_type.classes_)))
source_mapping = dict(zip(le_source.classes_, le_source.transform(le_source.classes_)))

print("Material Type Encoding:")
print(type_mapping)
print("\nSource Type Encoding:")
print(source_mapping)

# ── One-Hot Encoding for source_type (for ML models) ─────────
source_ohe = pd.get_dummies(df_materials['source_type'], prefix='source')
df_materials = pd.concat([df_materials, source_ohe], axis=1)

print("\n✅ Encoding complete. New columns added:")
print([c for c in df_materials.columns if 'enc' in c or 'source_' in c])

## 7. Normalization (Min-Max Scaling)

In [ ]:
scaler = MinMaxScaler()

cols_to_scale = ['strength_rating', 'weight_capacity_kg', 'biodegradability_score',
                 'co2_emission_score', 'recyclability_percentage',
                 'cost_per_kg_usd', 'decomposition_time_days', 'moisture_resistance']

scaled_col_names = [f"{c}_scaled" for c in cols_to_scale]
df_materials[scaled_col_names] = scaler.fit_transform(df_materials[cols_to_scale])

print("✅ Min-Max Scaling applied. Scaled columns:")
display(df_materials[scaled_col_names].describe().T.round(4))

## 8. Feature Engineering

Three composite indices are derived:
| Index | Formula | Interpretation |
|---|---|---|
| **CO₂ Impact Index** | `1 - co2_emission_score_scaled` | Higher = lower carbon footprint |
| **Cost Efficiency Index** | `1 - cost_per_kg_usd_scaled` | Higher = more cost-effective |
| **Material Suitability Score** | Weighted average of 5 scaled features | Higher = more sustainable overall |

In [ ]:
# ── CO₂ Impact Index ─────────────────────────────────────────────────────
# Inverted: lower CO₂ emission → higher index (better)
df_materials['co2_impact_index'] = (1 - df_materials['co2_emission_score_scaled']).round(4)

# ── Cost Efficiency Index ────────────────────────────────────────────────
# Inverted: lower cost → higher efficiency index
df_materials['cost_efficiency_index'] = (1 - df_materials['cost_per_kg_usd_scaled']).round(4)

# ── Material Suitability Score ───────────────────────────────────────────
# Weighted composite of key sustainability indicators
# Weights sum to 1.0
W_BIODEG      = 0.30   # Biodegradability (most important sustainability factor)
W_CO2         = 0.25   # CO₂ Impact (climate impact)
W_RECYCLE     = 0.20   # Recyclability (circular economy)
W_STRENGTH    = 0.15   # Strength (functional suitability)
W_MOISTURE    = 0.10   # Moisture Resistance (protection capability)

df_materials['material_suitability_score'] = (
    W_BIODEG   * df_materials['biodegradability_score_scaled'] +
    W_CO2      * df_materials['co2_impact_index'] +
    W_RECYCLE  * df_materials['recyclability_percentage_scaled'] +
    W_STRENGTH * df_materials['strength_rating_scaled'] +
    W_MOISTURE * df_materials['moisture_resistance_scaled']
).round(4)

print("✅ Feature Engineering complete.")
print()
print("New features added:")
print("  → co2_impact_index          (0–1, higher = better)")
print("  → cost_efficiency_index     (0–1, higher = better)")
print("  → material_suitability_score (0–1, higher = more suitable)")
print()
display(df_materials[['material_name', 'co2_impact_index',
                       'cost_efficiency_index',
                       'material_suitability_score']].sort_values(
                           'material_suitability_score', ascending=False).head(10))

### 8.1 Feature Correlation Heatmap

In [ ]:
feature_cols = ['biodegradability_score', 'co2_emission_score', 'recyclability_percentage',
               'cost_per_kg_usd', 'strength_rating', 'moisture_resistance',
               'co2_impact_index', 'cost_efficiency_index', 'material_suitability_score']

corr = df_materials[feature_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap — EcoPackAI', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Heatmap saved as 'correlation_heatmap.png'")

### 8.2 Top 15 Materials by Suitability Score

In [ ]:
top15 = df_materials.nlargest(15, 'material_suitability_score')[
    ['material_name', 'material_type', 'material_suitability_score',
     'co2_impact_index', 'cost_efficiency_index']
].reset_index(drop=True)

top15.index += 1  # Start ranking from 1
display(top15)

# Bar chart
plt.figure(figsize=(12, 6))
bars = plt.barh(top15['material_name'], top15['material_suitability_score'],
                color='teal', edgecolor='white')
plt.xlabel('Material Suitability Score')
plt.title('Top 15 Eco-Friendly Materials by Suitability Score', fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('top15_materials.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved as 'top15_materials.png'")

## 9. Save Cleaned & Engineered Data

In [ ]:
# Save enriched dataset as CSV
df_materials.to_csv('eco_materials_cleaned.csv', index=False)
print(f"✅ Cleaned dataset saved: eco_materials_cleaned.csv ({len(df_materials)} rows, {len(df_materials.columns)} cols)")

# Save to PostgreSQL as a new table
df_materials.to_sql(
    name='packaging_materials_processed',
    con=engine,
    if_exists='replace',
    index=False
)
print("✅ Processed data saved to PostgreSQL table: packaging_materials_processed")

## 10. Data Quality Report

In [ ]:
print("=" * 60)
print("MODULE 2 — DATA QUALITY REPORT")
print("=" * 60)
print(f"Total rows              : {len(df_materials)}")
print(f"Total columns           : {len(df_materials.columns)}")
print(f"Missing values          : {df_materials.isnull().sum().sum()}")
print(f"Duplicate rows          : {df_materials.duplicated().sum()}")
print()
print("── Engineered Features ──────────────────────────────────")
for feat in ['co2_impact_index', 'cost_efficiency_index', 'material_suitability_score']:
    print(f"  {feat:<35} min={df_materials[feat].min():.4f}  max={df_materials[feat].max():.4f}  mean={df_materials[feat].mean():.4f}")
print()
print("── Encoded Columns ──────────────────────────────────────")
enc_cols = [c for c in df_materials.columns if 'enc' in c or c.startswith('source_')]
print("  " + ", ".join(enc_cols))
print()
print("── Scaled Columns ───────────────────────────────────────")
scaled_cols = [c for c in df_materials.columns if '_scaled' in c]
print("  " + ", ".join(scaled_cols))
print()
print("✅ Module 2 Complete. Ready for Milestone 2 — ML Modelling.")